### Load Data
*Loading the raw file and working on a copy, so the original is never changed.*


In [1]:
import pandas as pd
import numpy as np

raw = pd.read_csv("../data/raw/ecommerce_orders_raw.csv")
df = raw.copy()
print("Start:", df.shape)

Start: (5020, 28)


**Result:** 5,020 rows and 28 columns, same as in Step 1.

### Remove Extra Spaces
*Trimming leading and trailing spaces in all text columns first, so that values such as ` m ` and `Germany ` match their clean versions.*

In [2]:
for col in df.select_dtypes(exclude="number").columns:
    df[col] = df[col].str.strip()

**Result:** Spaces removed from all text columns. In Step 1, `Gender` (820 rows), `Country` (458 rows) and `Customer_Name` (8 rows) were affected.

### Verifying that no spaces remain.


In [3]:
for c in df.select_dtypes(exclude="number").columns:
    assert df[c].dropna().str.contains(r"^\s|\s$").sum() == 0

   **Check:** After trimming, no text column has leading or trailing spaces (the assertion passed).

### Convert Order_Date
*`Order_Date` has 3 text formats. Each format is parsed separately and the results are combined. The slash format is day/month/year (confirmed in Step 1: the first part is sometimes greater than 12, the second never).*

In [4]:
# errors="coerce" turns a value that does not match the format into NaT instead of raising an error
# fillna chains the three formats: a row missed by one format is picked up by the next

iso = pd.to_datetime(df["Order_Date"], format="%Y-%m-%d", errors="coerce")
dmy = pd.to_datetime(df["Order_Date"], format="%d/%m/%Y", errors="coerce")
mon = pd.to_datetime(df["Order_Date"], format="%b %d, %Y", errors="coerce")
df["Order_Date"] = iso.fillna(dmy).fillna(mon)

print("Failed:", df["Order_Date"].isna().sum())
print(df["Order_Date"].min(), df["Order_Date"].max())

Failed: 0
2023-01-01 00:00:00 2025-12-31 00:00:00


**Result:** All dates converted (0 failed). The data covers 2023-01-01 to 2025-12-31.